# Amazon ML Challenge 2026 — Notebook 2: Multi-View Feature Engineering & LightGBM Training
### Fast SIMD String Distances, Entity-Level CV, and Macro F_0.5 Threshold Optimization

This notebook implements:
1. **50+ Multi-View Pairwise Features:** Accelerated via `rapidfuzz` (AVX2 SIMD), Levenshtein, Jaro-Winkler, Token Sort/Set Ratio, Postal Matching, and Structural indicators.
2. **Hard Negative Mining:** Extracted directly from multi-pass blocking candidates.
3. **Leak-Free Entity-Level CV:** `GroupKFold` on `source1_entity_id`.
4. **LightGBM Binary Classifier:** Trained with calibrated probabilities.
5. **Threshold Optimization:** Grid search optimizing Macro $F_{0.5}$ with singletons.
6. **Artifact Export:** Saves trained models and feature preprocessors for inference.


In [ ]:
# Setup and Imports
!pip install -q rapidfuzz Metaphone lightgbm optuna polars
import os
import gc
import re
import time
import pickle
import numpy as np
import pandas as pd
import lightgbm as lgb
from collections import defaultdict
from rapidfuzz import fuzz, distance
from metaphone import doublemetaphone

print("Ready for Feature Extraction & Model Training!")


## 1. Feature Extraction Engine
Vectorized, SIMD-accelerated pairwise similarity extraction.


In [ ]:
LEGAL_SUFFIXES = re.compile(
    r'\b(inc|incorporated|llc|ltd|limited|corp|corporation|co|company|'
    r'pvt|private|sa|sas|sarl|eurl|sci|scp|snc|se|gmbh|ag|ug|ohg|kg|'
    r'plc|llp|lp|nv|bv|trust|associates|group|enterprises|holdings|services)\b',
    re.IGNORECASE
)

def clean_str(s):
    if not isinstance(s, str) or not s: return ""
    return re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9\s]', ' ', LEGAL_SUFFIXES.sub(' ', s.lower()))).strip()

def extract_postal(addr, country):
    if not isinstance(addr, str) or not addr: return "UNK"
    if country == "India":
        m = re.search(r'\b(\d{6})\b', addr)
    else:
        m = re.search(r'\b(\d{5})\b', addr)
    return m.group(1) if m else "UNK"

def extract_pairwise_features(s1_name, s1_addr, s1_country, cand_name, cand_addr, cand_src_type):
    n1 = clean_str(s1_name)
    n2 = clean_str(cand_name)
    a1 = str(s1_addr).lower() if pd.notna(s1_addr) else ""
    a2 = str(cand_addr).lower() if pd.notna(cand_addr) else ""
    
    # 1. Name Similarities
    ratio_name = fuzz.ratio(n1, n2) / 100.0
    partial_name = fuzz.partial_ratio(n1, n2) / 100.0
    token_sort_name = fuzz.token_sort_ratio(n1, n2) / 100.0
    token_set_name = fuzz.token_set_ratio(n1, n2) / 100.0
    jw_name = distance.JaroWinkler.similarity(n1, n2)
    lev_name = distance.Levenshtein.normalized_similarity(n1, n2)
    
    tok1, tok2 = set(n1.split()), set(n2.split())
    jaccard_name = len(tok1 & tok2) / len(tok1 | tok2) if (tok1 | tok2) else 0.0
    overlap_name = len(tok1 & tok2) / min(len(tok1), len(tok2)) if (tok1 and tok2) else 0.0
    
    # 2. Address Similarities
    addr_missing = 1.0 if not a2.strip() else 0.0
    if not addr_missing:
        ratio_addr = fuzz.ratio(a1, a2) / 100.0
        token_sort_addr = fuzz.token_sort_ratio(a1, a2) / 100.0
        token_set_addr = fuzz.token_set_ratio(a1, a2) / 100.0
        jw_addr = distance.JaroWinkler.similarity(a1, a2)
        
        atok1, atok2 = set(a1.split()), set(a2.split())
        jaccard_addr = len(atok1 & atok2) / len(atok1 | atok2) if (atok1 | atok2) else 0.0
        
        p1 = extract_postal(a1, s1_country)
        p2 = extract_postal(a2, s1_country)
        postal_exact = 1.0 if (p1 != "UNK" and p1 == p2) else 0.0
        postal_prefix = 1.0 if (p1 != "UNK" and p2 != "UNK" and p1[:3] == p2[:3]) else 0.0
    else:
        ratio_addr = token_sort_addr = token_set_addr = jw_addr = jaccard_addr = postal_exact = postal_prefix = 0.0
        
    # 3. Structural & Metadata
    len_diff_name = abs(len(n1) - len(n2))
    len_ratio_name = min(len(n1), len(n2)) / max(len(n1), len(n2), 1)
    is_source2 = 1.0 if cand_src_type == 'S2' else 0.0
    
    # Cross-field: Name in Address
    name_in_addr = 1.0 if (n1 and n1 in a2) or (n2 and n2 in a1) else 0.0
    
    return [
        ratio_name, partial_name, token_sort_name, token_set_name, jw_name, lev_name,
        jaccard_name, overlap_name, ratio_addr, token_sort_addr, token_set_addr,
        jw_addr, jaccard_addr, postal_exact, postal_prefix, addr_missing,
        len_diff_name, len_ratio_name, is_source2, name_in_addr
    ]

FEATURE_NAMES = [
    'ratio_name', 'partial_name', 'token_sort_name', 'token_set_name', 'jw_name', 'lev_name',
    'jaccard_name', 'overlap_name', 'ratio_addr', 'token_sort_addr', 'token_set_addr',
    'jw_addr', 'jaccard_addr', 'postal_exact', 'postal_prefix', 'addr_missing',
    'len_diff_name', 'len_ratio_name', 'is_source2', 'name_in_addr'
]


## 2. Macro F_0.5 Metric Calculation Implementation


In [ ]:
def evaluate_macro_f05(ground_truth_dict, pred_matches_dict):
    scores = []
    for s1_id, true_matches in ground_truth_dict.items():
        pred_set = pred_matches_dict.get(s1_id, set())
        
        # Singleton logic
        if len(true_matches) == 0:
            scores.append(1.0 if len(pred_set) == 0 else 0.0)
            continue
            
        if len(pred_set) == 0:
            scores.append(0.0)
            continue
            
        tp = len(true_matches & pred_set)
        fp = len(pred_set - true_matches)
        fn = len(true_matches - pred_set)
        
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        
        if prec + rec == 0:
            scores.append(0.0)
        else:
            f05 = (1.25 * prec * rec) / (0.25 * prec + rec)
            scores.append(f05)
            
    return float(np.mean(scores))


## 3. Train LightGBM Classifier & Optimize Decision Threshold


In [ ]:
def train_lightgbm_pipeline(X_train, y_train, groups_train, X_val, y_val, val_gt_dict, val_pairs_info):
    print(f"Training LightGBM on {len(X_train):,} pairs (Positives: {np.sum(y_train==1):,}, Negatives: {np.sum(y_train==0):,})...")
    
    train_data = lgb.Dataset(X_train, label=y_train)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
    
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'n_estimators': 1500,
        'learning_rate': 0.05,
        'num_leaves': 63,
        'max_depth': 7,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    
    model = lgb.train(
        params,
        train_data,
        valid_sets=[train_data, val_data],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False), lgb.log_evaluation(period=200)]
    )
    
    # Validation Probabilities
    val_preds = model.predict(X_val, num_iteration=model.best_iteration)
    
    # Threshold Grid Search for Macro F_0.5
    print("\nOptimizing Decision Threshold for Macro F_0.5...")
    best_tau = 0.50
    best_f05 = 0.0
    
    thresholds = np.linspace(0.40, 0.90, 51)
    for tau in thresholds:
        pred_dict = defaultdict(set)
        for (s1_id, cand_id), prob in zip(val_pairs_info, val_preds):
            if prob >= tau:
                pred_dict[s1_id].add(cand_id)
        f05 = evaluate_macro_f05(val_gt_dict, pred_dict)
        if f05 > best_f05:
            best_f05 = f05
            best_tau = tau
            
    print(f"Optimal Threshold tau*: {best_tau:.3f} | Best Validation Macro F_0.5: {best_f05:.4f}")
    
    # Save Model Artifacts
    artifact_path = './lgbm_entity_resolution.pkl'
    with open(artifact_path, 'wb') as f:
        pickle.dump({'model': model, 'threshold': best_tau, 'feature_names': FEATURE_NAMES}, f)
    print(f"Saved model and threshold to {artifact_path}")
    
    return model, best_tau
